In [1]:
import os
import shutil

# Setup: Check for repo and enforce working directory
repo_path = "/kaggle/working/rl_sf"
if not os.path.exists(repo_path):
    print("--> Repository missing. Cloning now...")
    !git clone -b optimization https://github.com/flaviogeuforbio/rl-with-sf-for-mujoco {repo_path}

# Change directory explicitly to where the script lives
%cd {repo_path}

--> Repository missing. Cloning now...
Cloning into '/kaggle/working/rl_sf'...
remote: Enumerating objects: 3916, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 3916 (delta 2), reused 6 (delta 2), pack-reused 3910 (from 3)
Receiving objects: 100% (3916/3916), 828.00 MiB | 32.91 MiB/s, done.
Resolving deltas: 100% (765/765), done.
Updating files: 100% (2430/2430), done.
/kaggle/working/rl_sf


In [2]:
import torch
print("torch:", torch.version)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch: <module 'torch.version' from '/usr/local/lib/python3.12/dist-packages/torch/version.py'>
cuda available: False
device: CPU


In [3]:
%ls

ActorCritic.py                        render_agent.py
args_plot_average_results_walker.txt  render_args_cheetah.txt
args_render_agent.txt                 render_args_walker_transfer.txt
artifacts/                            requirements.txt
diagnose_feature_scales.py            run_behavioral_diagnostics_mod.py
diagnose_psi.py                       run_behavioral_diagnostics.py
diagnose_rollout_dynamics.py          run_psi_diagnostic_mod.py
OLD_2_train_cheetah_walker.py         run_psi_diagnostic.py
OLD_train_cheetah_walker.py           train_cheetah_walker.py
plot_average_results.py               train_sf_ddpg.py
PlotResults.py                        transfer_vs_scratch_comparison.pdf
plot_rollout_timeseries.py            transfer_vs_scratch_comparison.png
__pycache__/                          utils.py
QuickPlot.py                          zero_shot_eval.py
quick_test.py


In [4]:
!python -m pip install mujoco

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 61.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 12.8 MB/s eta 0:00:00


In [ ]:
import os
import shutil

# --- PHASE 1: INITIAL RUN ---
seed = 1
folder_name = f"seed_{seed}_transfer" # Dynamic folder name

# For a quick sanity check:
# STEPS_PER_PHASE = "4000"  # target
# RUN_STEPS_LIMIT = "10000"   # chunk limit
# SAVE_FREQ = "2000"         # Synchronized with limit

STEPS_PER_PHASE = "4000000"  # 4M target
RUN_STEPS_LIMIT = "1000000"   # chunk limit
SAVE_FREQ = "100000"         # Synchronized with limit
GAMMA_VAL = "0.99"
LAMBDA_Q = "1.0" 
LAMBDA_VEC = "1.0" 
#RESUME_DIR = "/kaggle/input/datasets/adrianoarceri/checkpoint" # change the last name accoording to how you name the dataset, and the previous folder is your username
# Here I don't need the resume_dir!!

RUN_NAME_SEQ = f"Walker_transfer_gamma_{GAMMA_VAL.replace('.', '_')}_lq_{LAMBDA_Q.replace('.', '_')}_lvec_{LAMBDA_VEC.replace('.', '_')}_stepsxphase_{STEPS_PER_PHASE}"

# Create the dynamic directory
kaggle_output_folder = f"/kaggle/working/{folder_name}"
os.makedirs(kaggle_output_folder, exist_ok=True)

local_path_seq = f"/kaggle/working/rl_sf/artifacts/walker/{RUN_NAME_SEQ}"

print(f"\n--- EXECUTING SEED {seed} ---")

print("-> Running Sequential Training...")
!python -u train_cheetah_walker.py --steps_per_phase {STEPS_PER_PHASE} --save_freq {SAVE_FREQ} --baseline --run_name {RUN_NAME_SEQ} --gamma {GAMMA_VAL} --lambda_q {LAMBDA_Q} --lambda_vec {LAMBDA_VEC} --seed {seed} --run_steps_limit {RUN_STEPS_LIMIT}

if os.path.exists(local_path_seq):
    final_dest_seq = os.path.join(kaggle_output_folder, RUN_NAME_SEQ)
    shutil.copytree(local_path_seq, final_dest_seq, dirs_exist_ok=True)
    print(f"--> Sequential data saved in: {final_dest_seq}")

print("\n--> Zipping results for download...")
%cd /kaggle/working/
!zip -r {folder_name}.zip {folder_name}/
%cd /kaggle/working/rl_sf


--- EXECUTING SEED 1 ---
-> Running Sequential Training...
Training SF-DDPG...
--- Starting Phase 0 (Cheetah) ---
--> [SAVE] Checkpoint saved at Phase 0, Step 2000
--- Starting Phase 1 (Walker) ---
Step: 196 | Episodes: 10 | Avg Return: 18.72 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 383 | Episodes: 20 | Avg Return: 17.68 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 554 | Episodes: 30 | Avg Return: 16.23 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 737 | Episodes: 40 | Avg Return: 17.30 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 950 | Episodes: 50 | Avg Return: 20.45 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 1105 | Episodes: 60 | Avg Return: 14.49 | C Loss: 7.2995 | Q Loss: 5.1827 | Vec Loss: 2.1168
Step: 1225 | Episodes: 70 | Avg Return: 10.93 | C Loss: 3.5133 | Q Loss: 2.8166 | Vec Loss: 0.6968
Step: 1345 | Episodes: 80 | Avg Return: 10.93 | C Loss: 1.7300 | Q Loss: 1.3188 | Vec Loss: 0.4112
Step: 1465 | E